# Guardrail and retrieval walkthrough

Notes to myself on how the reply pipeline actually behaves, with the
decisions I made and the ones I got wrong. Run from the repo root.

The point of this notebook is that the routing layer is where every bug
in this project has been so far, and reading the code doesn't catch them.
Running inputs through it does.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else '.')

# Dummy creds so app.config imports. Nothing here hits the network or the DB.
for k, v in {
    'SUPABASE_URL': 'https://test.supabase.co', 'SUPABASE_SERVICE_KEY': 'test',
    'OPENAI_API_KEY': 'sk-test', 'TWILIO_ACCOUNT_SID': 'ACtest',
    'TWILIO_AUTH_TOKEN': 'test', 'TWILIO_FROM_NUMBER': '+15550000000',
}.items():
    os.environ.setdefault(k, v)

from app.kb.loader import load_kb, search, match_restricted, wants_human, _tokens
from app.phone import normalize_phone, InvalidPhoneError

kb = load_kb()
print(f'{len(kb.entries)} entries, {len(kb.restricted)} restricted topics')


## 1. What's in the KB

Everything the agent is allowed to assert. If it isn't here it escalates.


In [ ]:
for e in kb.entries:
    print(f'{e.id:24} {e.topic:18} {len(e.triggers)} triggers')
print()
for r in kb.restricted:
    print(f'{r.id:24} -> {r.reason}')


## 2. Phone normalization

This was the first thing that broke in production. The old version treated
any leading `+` as proof the number was already full E.164, so a Toronto
number typed `+416 822 6186` got stored as `+4168226186` and Twilio
rejected it with error 21211 — it reads the leading 4 as a country code.

It surfaced as a 500 at send time. The worse damage was at CSV import,
where it failed silently.


In [ ]:
cases = ['+416 822 6186', '416 822 6186', '(416) 822-6186', '4168226186',
         '1-416-822-6186', '+14168226186', '+44 20 7946 0958', '555-1234', '12', '']

for raw in cases:
    try:
        print(f'{raw!r:22} -> {normalize_phone(raw)}')
    except InvalidPhoneError:
        print(f'{raw!r:22} -> rejected')


`555-1234` rejecting is deliberate and I only found it by writing the test.
Seven digits fell inside my 'assume it has a country code' branch and came
out as `+5551234`, which looks plausible and isn't. A bare 7-digit string
is a local number missing its area code. Floor is 8 now.


## 3. Restricted topics

Pricing, contracts, legal. These never reach a model — whether 'how much
does it cost' is close enough to answer shouldn't depend on a sampled
token. Plain string matching, checked before anything else runs.


In [ ]:
probes = ['how much does it cost', 'what do you charge', 'is there a contract',
          'can I cancel anytime', 'do you record calls', 'does it integrate with Jobber',
          'is this a robot', 'how does it work']

for m in probes:
    r = match_restricted(m)
    print(f'{m!r:34} -> {r.id if r else "allowed"}')


### The substring bug

Matching used to be `trigger in message`. That fires on any word that
merely contains a trigger. A prospect texted 'You like feet?' and got back
a holding reply about getting them exact pricing, because `fee` is inside
`feet`.

Same class of failure: `rate` inside `accurate`, `api` inside `rapid`,
`cost` inside `costume`. Fixed with word boundaries.


In [ ]:
import re

def old_match(message):
    low = message.lower()
    for r in kb.restricted:
        if any(t in low for t in r.triggers):
            return r.id
    return None

collisions = ['do you like feet', 'that was accurate', 'we need a rapid response',
              'he wore a costume', 'how much does it cost']

print(f'{"message":32} {"old":18} new')
for m in collisions:
    new = match_restricted(m)
    print(f'{m:32} {str(old_match(m)):18} {new.id if new else None}')


Last row is the control — a real pricing question still matches both ways.

## 4. Retrieval

Lexical, not vector. The property I care about is that an off-topic message
returns **nothing**, because empty is what tells the agent it may not make
a factual claim. Cosine similarity always returns a nearest neighbour, so
I'd need an arbitrary threshold to reproduce that.


In [ ]:
queries = ['what is voicecaptures', 'is this a robot', 'are you alive',
           'how does this work', 'do you have a free trial',
           "what's the weather like in the uk", 'who won the game', 'lol', 'asdfgh']

for q in queries:
    hits = search(q)
    print(f'{q!r:36} -> {[e.id for e in hits] or "nothing"}')


### Where lexical retrieval loses

'How does this work' is the most common question a prospect can ask and it
originally matched nothing. Everything in it is a stopword except 'work',
and no trigger contained that word.


In [ ]:
for q in ['how does this work', 'what is voicecaptures', "what's the weather like in the uk"]:
    print(f'{q!r:36} -> {sorted(_tokens(q))}')


I patched it by adding trigger phrases. That works but it doesn't
generalise — I can't enumerate English, and the next prospect writes
'walk me through it'.

Two options and I haven't picked yet. Keep patching as gaps appear, which
is cheap and fails safe because a miss escalates to me rather than
producing a wrong answer. Or add embeddings as a fallback only when
lexical returns nothing, which catches paraphrase but reintroduces the
threshold I avoided.

The honest answer is I don't know the miss rate yet. That's what the eval
set is for — measure it, then decide.


### An earlier miss, worth keeping in here

The tokenizer used to keep apostrophes, so `what's` never matched the
stopword `what` and survived as a content word. It then scored against any
trigger phrased as a question, which meant the weather query pulled back
the product overview.

I'd tested this by hand and it passed — because I typed it without the
apostrophe. A test caught it in under a second.


In [ ]:
def old_tokens(text):
    from app.kb.loader import _STOPWORDS
    return {w for w in re.findall(r"[a-z']+", text.lower())
            if w not in _STOPWORDS and len(w) > 2}

q = "what's the weather like in the uk"
print('old:', sorted(old_tokens(q)))
print('new:', sorted(_tokens(q)))


## 5. Full routing decision

Same order the live path uses, minus the model calls. Cheapest and most
certain check first.


In [ ]:
def route(message):
    if wants_human(message):
        return 'handoff', kb.handoff_holding_reply
    r = match_restricted(message)
    if r:
        return f'restricted:{r.id}', r.holding_reply
    hits = search(message)
    if not hits:
        return 'no_kb_match', 'classifier decides, agent may assert nothing'
    return 'answerable', ', '.join(e.id for e in hits)

for m in ['how much does it cost', 'can I talk to someone', 'is this a robot',
          'how does this work', "what's the weather in the uk", 'do you like feet',
          'is there a contract', 'ok sure']:
    decision, detail = route(m)
    print(f'{m!r:36} {decision:22} {detail[:44]}')


'do you like feet' landing in `no_kb_match` rather than `restricted:pricing`
is the substring fix working. The agent gets no entries, so it can't make a
claim, and it replies with the scope-rule redirect.

## 6. Holding replies

A blocked message still gets an immediate acknowledgement. Before these
existed, a pricing question — the highest-intent message in the funnel —
got total silence while it sat in my review queue.

They're constants from the YAML, never model output, and they contain no
factual claim. That's the whole reason they're safe to send automatically
on a path where the agent itself isn't trusted to speak.


In [ ]:
for r in kb.restricted:
    has_digit = any(c.isdigit() for c in r.holding_reply)
    print(f'{r.id:24} digits={has_digit}')
    print(f'  {r.holding_reply}')
print()
print('kb gap:', kb.kb_gap_holding_reply)
print('handoff:', kb.handoff_holding_reply)


The digit check is a test in the suite too. These bypass the grounding
guardrail, so a number appearing in one would be an unverified claim going
out unchecked. Easiest way to catch that drift is to assert no digits.

## 7. What's still open

- KB answers are placeholders I haven't verified against the real product.
  One test fails on this and should keep failing until it's done.
- No measurement of the scope classifier or the grounding judge. Both are
  models judging text and I have no idea how often they're wrong.
- Retrieval miss rate unknown. Two examples found by accident is not data.
